# 수상작 리뷰: ASK1 IC50 예측 — LightGBM + Optuna 튜닝 베이스라인


# 1. 데이터

- 데이터 출처: ChEMBL(ASK1, IC50) + PubChem(ASK1) 공개 데이터
- 데이터 크기: [train_df.shape 결과 입력] 관측치 (분자 수)
- 입력 데이터: 분자 구조 SMILES
- 종속변수: IC50 (nM) → pIC50로 변환하여 학습
- 분석 유형: 회귀

# 2. 주요 변수

- 구조 정보: Morgan Fingerprint (radius 2, 2048 bit)
- 물성 정보: RDKit Descriptor 전체 (분자량, logP, TPSA 등 200여 개)
- 최종 입력: Fingerprint + Descriptor를 이어 붙인 약 2,250차원 벡터

# 3. 데이터 특성

- IC50는 값의 범위가 매우 넓고 오른쪽으로 치우친 분포
- ChEMBL과 PubChem 두 출처를 합쳐 사용
- 같은 SMILES가 두 출처에 중복될 수 있음
- 해석 불가능한 SMILES는 특징 추출이 안 되어 제외됨
- 평가 점수: 0.4x(1-NRMSE) + 0.6xR²

# 4. 전처리 방향

- ChEMBL에서 Standard Type이 IC50인 행만 사용
- 컬럼명을 smiles, ic50_nM으로 통일 후 병합
- 숫자 변환 실패 및 결측 제거
- SMILES 기준 중복 제거 (먼저 나온 값 유지)
- IC50 > 0인 값만 사용
- pIC50 = 9 − log10(IC50) 로그 변환
- Descriptor 결측은 컬럼 평균으로 대체 후 StandardScaler 적용

# 5. 코드 흐름

- 데이터 로드·정제 → pIC50 변환 → 특징 생성 (Fingerprint + Descriptor)
- Optuna 50회 탐색 (목적함수: 5-Fold OOF 대회 점수)
- 최적 파라미터로 5-Fold 모델 학습 → 테스트 예측 평균
- pIC50를 IC50로 복원 → 제출 파일 생성
- 예측 불가 SMILES는 학습 IC50 평균으로 대체

# 6. 주요 코드 분석

- pIC50 변환: 치우친 IC50 분포를 로그 스케일로 바꿔 학습을 안정화하고, 예측 후 역변환
- 특징 엔지니어링: Fingerprint(부분 구조)와 Descriptor(물성)를 함께 사용해 구조와 성질을 동시에 반영
- 목적함수 설계: RMSE가 아니라 대회 평가식을 Optuna의 최대화 대상으로 직접 사용하여 리더보드와 튜닝 방향을 일치시킴
- OOF 평가: 5-Fold 전체 OOF 예측으로 점수를 한 번에 계산해 Fold별 평균보다 안정적으로 평가
- 탐색 공간: learning_rate, num_leaves, max_depth, feature_fraction, bagging_fraction, bagging_freq, min_child_samples
- 트리 수 결정: n_estimators=2000과 early_stopping(100) 조합으로 자동 결정
- 앙상블: Fold별 5개 모델의 예측을 평균하여 분산 감소

In [4]:
!pip install -q rdkit optuna lightgbm
# 데이터 파일(ChEMBL_ASK1(IC50).csv, Pubchem_ASK1.csv, test.csv, sample_submission.csv)을
# 왼쪽 파일 탭에 업로드하거나 드라이브를 마운트한 뒤 경로를 맞춰주세요.

KeyboardInterrupt: 

### 2-1. 설정 및 라이브러리

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_squared_error
import optuna

CFG = {
    'NBITS': 2048,
    'SEED': 42,
    'N_SPLITS': 5,
    'N_TRIALS': 50
}

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG['SEED'])

### 2-2. 데이터 로드 및 전처리

In [ ]:
def load_and_preprocess_data():
    try:
        chembl = pd.read_csv("./ChEMBL_ASK1(IC50).csv", sep=';')
        pubchem = pd.read_csv("./Pubchem_ASK1.csv")
    except FileNotFoundError as e:
        print(f"Error: {e}. Make sure data files are in the current directory.")
        return None

    chembl.columns = chembl.columns.str.strip().str.replace('"', '')
    chembl = chembl[chembl['Standard Type'] == 'IC50']
    chembl = chembl[['Smiles', 'Standard Value']].rename(columns={'Smiles': 'smiles', 'Standard Value': 'ic50_nM'})
    chembl['ic50_nM'] = pd.to_numeric(chembl['ic50_nM'], errors='coerce')

    pubchem = pubchem[['SMILES', 'Activity_Value']].rename(columns={'SMILES': 'smiles', 'Activity_Value': 'ic50_nM'})
    pubchem['ic50_nM'] = pd.to_numeric(pubchem['ic50_nM'], errors='coerce')

    df = pd.concat([chembl, pubchem], ignore_index=True).dropna(subset=['smiles', 'ic50_nM'])
    df = df.drop_duplicates(subset='smiles').reset_index(drop=True)
    df = df[df['ic50_nM'] > 0]
    return df

def IC50_to_pIC50(ic50_nM): return 9 - np.log10(ic50_nM)
def pIC50_to_IC50(pIC50): return 10**(9 - pIC50)

train_df = load_and_preprocess_data()
train_df['pIC50'] = IC50_to_pIC50(train_df['ic50_nM'])
print(train_df.shape)
train_df.head()

### 2-3. 특징 생성 함수 (Fingerprint + RDKit Descriptor)

In [ ]:
def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=CFG['NBITS'])
        arr = np.zeros((1,))
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr
    return None

def calculate_rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.full((len(Descriptors._descList),), np.nan)
    descriptors = [desc_func(mol) for _, desc_func in Descriptors._descList]
    return np.array(descriptors)

### 2-4. 평가 지표 및 Optuna 목적함수

In [ ]:
def get_score(y_true_ic50, y_pred_ic50, y_true_pic50, y_pred_pic50):
    # 최신 scikit-learn에서는 squared=False 옵션이 제거되어 np.sqrt로 대체
    rmse = np.sqrt(mean_squared_error(y_true_ic50, y_pred_ic50))
    nrmse = rmse / (np.max(y_true_ic50) - np.min(y_true_ic50))
    A = 1 - min(nrmse, 1)
    B = r2_score(y_true_pic50, y_pred_pic50)
    return 0.4 * A + 0.6 * B

def objective(trial, X, y):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'verbose': -1, 'n_jobs': -1,
        'seed': CFG['SEED'], 'boosting_type': 'gbdt', 'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
    }

    kf = KFold(n_splits=CFG['N_SPLITS'], shuffle=True, random_state=CFG['SEED'])
    oof_preds = np.zeros(len(X))

    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        model = lgb.LGBMRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
                  eval_metric='rmse', callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_preds[val_idx] = model.predict(X_val)

    y_ic50_true = pIC50_to_IC50(y)
    oof_ic50_preds = pIC50_to_IC50(oof_preds)
    return get_score(y_ic50_true, oof_ic50_preds, y, oof_preds)

### 2-5. 특징 행렬 생성

In [ ]:
train_df['fingerprint'] = train_df['smiles'].apply(smiles_to_fingerprint)
train_df['descriptors'] = train_df['smiles'].apply(calculate_rdkit_descriptors)
train_df.dropna(subset=['fingerprint', 'descriptors'], inplace=True)

desc_stack = np.stack(train_df['descriptors'].values)
desc_mean = np.nanmean(desc_stack, axis=0)
desc_stack = np.nan_to_num(desc_stack, nan=desc_mean)

scaler = StandardScaler()
desc_scaled = scaler.fit_transform(desc_stack)
fp_stack = np.stack(train_df['fingerprint'].values)

X = np.hstack([fp_stack, desc_scaled])
y = train_df['pIC50'].values
print("X:", X.shape, "y:", y.shape)

### 2-6. Optuna 하이퍼파라미터 탐색

In [ ]:
study = optuna.create_study(direction='maximize', study_name='lgbm_tuning')
study.optimize(lambda trial: objective(trial, X, y), n_trials=CFG['N_TRIALS'])

print(f"Best Score: {study.best_value:.4f}")
print("Best Parameters:", study.best_params)

### 2-7. 최종 학습(5-Fold 앙상블) 및 제출 파일 생성

In [ ]:
best_params = {'objective': 'regression', 'metric': 'rmse', 'verbose': -1, 'n_jobs': -1,
               'seed': CFG['SEED'], 'boosting_type': 'gbdt', 'n_estimators': 2000}
best_params.update(study.best_params)

test_df = pd.read_csv("./test.csv")
test_df['fingerprint'] = test_df['Smiles'].apply(smiles_to_fingerprint)
test_df['descriptors'] = test_df['Smiles'].apply(calculate_rdkit_descriptors)

valid_test_mask = test_df['fingerprint'].notna() & test_df['descriptors'].notna()
fp_test_stack = np.stack(test_df.loc[valid_test_mask, 'fingerprint'].values)
desc_test_stack = np.stack(test_df.loc[valid_test_mask, 'descriptors'].values)
desc_test_stack = np.nan_to_num(desc_test_stack, nan=desc_mean)
desc_test_scaled = scaler.transform(desc_test_stack)
X_test = np.hstack([fp_test_stack, desc_test_scaled])

kf = KFold(n_splits=CFG['N_SPLITS'], shuffle=True, random_state=CFG['SEED'])
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"--- Training Fold {fold+1}/{CFG['N_SPLITS']} ---")
    model = lgb.LGBMRegressor(**best_params)
    model.fit(X[train_idx], y[train_idx])          # 검증셋/early stopping 없이 n_estimators=2000 그대로 학습
    test_preds += model.predict(X_test) / CFG['N_SPLITS']

submission_df = pd.read_csv("./sample_submission.csv")
pred_df = pd.DataFrame({'ID': test_df.loc[valid_test_mask, 'ID'],
                        'ASK1_IC50_nM': pIC50_to_IC50(test_preds)})
submission_df = submission_df[['ID']].merge(pred_df, on='ID', how='left')
# ⚠️ pandas 최신 버전에서 inplace fillna 경고가 나므로 대입 방식으로 변경
submission_df['ASK1_IC50_nM'] = submission_df['ASK1_IC50_nM'].fillna(train_df['ic50_nM'].mean())
submission_df.to_csv("lgbm_tuned_submission.csv", index=False)
submission_df.head()

## 새롭게 알게 된 점 / 아쉬운 점

### 새롭게 알게 된 점
- **타깃을 로그 변환(pIC50)** 하는 것이 활성 예측의 표준적인 접근이다
- **평가 지표 자체를 Optuna 목적함수로 사용**하면 튜닝 결과가 리더보드와 일관되게 움직인다
- 분자 데이터는 **Fingerprint(구조)와 Descriptor(물성)를 함께** 쓰는 구성이 간단하면서도 강력한 베이스라인이 된다
- OOF 예측으로 점수를 계산하고, Fold별 모델을 평균 내는 **CV 앙상블** 구조

### 아쉬운 점 및 개선 아이디어
1. **튜닝과 최종 학습의 불일치**: 튜닝 때는 `early_stopping`으로 최적 트리 수를 찾았지만, 최종 학습에서는 검증셋 없이 `n_estimators=2000`을 그대로 사용한다. 과적합 위험이 있으므로 Fold별 `best_iteration`을 쓰거나 최종 학습에도 early stopping을 적용하는 것이 일관적이다.
2. **OOF 점수의 낙관적 편향**: early stopping에 검증 Fold를 사용하므로 OOF 점수가 실제보다 약간 높게 나올 수 있다.
3. **중복 처리 방식**: `smiles` 기준 `drop_duplicates`는 먼저 나온 값(ChEMBL)만 남긴다. 같은 분자가 두 출처에 있을 때 값이 다를 수 있으므로 **중앙값/평균으로 집계**하거나, SMILES를 canonical 형태로 통일한 뒤 중복 제거하는 편이 더 정확하다.
4. **데이터 누수 가능성**: `StandardScaler`와 평균 대체를 전체 학습 데이터로 fit한 뒤 CV를 수행한다. 영향은 작지만 엄밀히는 Fold 안에서 fit해야 한다. (트리 모델에서는 스케일링 자체가 불필요함)
5. **분할 방식**: 랜덤 KFold는 유사한 분자가 train/val에 섞여 점수가 높게 나올 수 있다. **Scaffold split**이 일반화 성능을 더 잘 반영한다.
6. **결측 대체**: 예측 불가 샘플을 학습 IC50의 산술 평균으로 채우면 편향된 값(우측으로 치우친 분포)이 들어간다. 중앙값 또는 pIC50 평균의 역변환이 더 적절하다.
7. **모델 다양성 부족**: LightGBM 단일 모델, CatBoost/XGBoost, 다른 Fingerprint(MACCS, AtomPair), GNN 등을 섞은 앙상블로 확장할 여지가 있다.
8. **Descriptor 이상값(inf) 미처리**: 일부 RDKit 값이 `inf`일 수 있어 사전 점검이 필요하다.